# TN1 — Kiến trúc: TCN so với DS-TCN so với LSTM

Đổi **đúng một biến** so với TN0: thay LSTM của MobiVital bằng TCN. Mọi thứ khác giữ nguyên cấu hình tác giả công bố.

## Câu hỏi

**Thu nhỏ model 90–96% thì còn dự báo tốt hơn LSTM không?**

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| LSTM (MobiVital) | hidden 352 | 1.502.713 | — |
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

## Ba notebook, chạy theo thứ tự này

| | notebook | chạy gì | thời gian |
|---|---|---|---|
| 1 | `TN1_LSTM.ipynb` | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 h |
| 1 | `TN1_TCN_DSTCN_model_selection.ipynb` | TCN-64 và DS-TCN-64: 4 fold CV | ~2.2 h |
| 2 | **`TN1_final_evaluation.ipynb`** ← đang mở | gộp kết quả, so sánh, GHIJ cho kiến trúc thắng | ~1.5 h |

Hai notebook đầu chạy **song song** ở hai phiên Colab khác nhau, không cần chờ nhau. Mỗi cái nén kết quả ra một tệp zip riêng trên Drive:

```
tn1_lstm.zip    runs/tn1/ (phần lstm) · runs/tn1_ghij/ · summary.csv
tn1_tcn.zip     runs/tn1/ (phần tcn và ds_tcn) · summary.csv
```

Notebook này bung cả hai rồi gộp lại.

## Giao thức

Bốn fold cố định trên `A B C D E F K L`. **`G H I J` chỉ đụng ở mục 4**, và chỉ để làm mốc kiểm chứng — quyết định chọn kiến trúc phải dựa trên `cv_score`, xem `docs/PROTOCOL.md`.


## 1. Chuẩn bị Colab


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


In [ ]:
!python scripts/restore_processed_data_on_drive.py


## 2. Gộp kết quả hai phiên

Hai phiên chạy độc lập nên mỗi phiên chỉ có dòng `summary.csv` của riêng mình. Tệp điểm từng buổi ghi thì không đè nhau — tên cấu hình đã phân biệt (`lstm_...`, `tcn_c64_...`, `ds_tcn_c64_...`).

Ô dưới bung cả hai zip rồi nối hai `summary.csv`, bỏ dòng tiêu đề lặp.


In [ ]:
import csv, os, subprocess
DRIVE = "/content/drive/MyDrive/mobivital"

for z in ["tn1_lstm.zip", "tn1_tcn.zip"]:
    assert os.path.exists(DRIVE + "/" + z), "thiếu " + z + " — phiên kia chạy xong chưa?"

# Bung ra hai chỗ riêng rồi mới gộp, để không cái nào đè summary.csv của cái kia.
for z, dich in [("tn1_lstm.zip", "/content/p1"), ("tn1_tcn.zip", "/content/p2")]:
    subprocess.run("rm -rf %s && mkdir -p %s && unzip -qo %s/%s -d %s" % (dich, dich, DRIVE, z, dich), shell=True)

subprocess.run("mkdir -p runs/tn1 runs/tn1_ghij", shell=True)
subprocess.run("cp -r /content/p1/tn1/. runs/tn1/ ; cp -r /content/p2/tn1/. runs/tn1/", shell=True)
subprocess.run("cp -r /content/p1/tn1_ghij/. runs/tn1_ghij/ 2>/dev/null", shell=True)

# Nối hai summary.csv, giữ một dòng tiêu đề, bỏ dòng trùng
rows, header, thay = [], None, set()
for p in ["/content/p1/summary.csv", "/content/p2/summary.csv"]:
    if not os.path.exists(p):
        continue
    r = list(csv.reader(open(p)))
    header = r[0]
    for dong in r[1:]:
        if dong and dong[0] not in thay:
            thay.add(dong[0]); rows.append(dong)

with open("runs/summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(header); w.writerows(rows)

print("gộp xong:", len(rows), "dòng trong runs/summary.csv")
for d in sorted(set(r[0] for r in rows)):
    print("   ", d)


## 3. So sánh

Hai bảng:

1. **`cv_score`** — điểm trung bình 4 fold của từng cấu hình, kèm chi tiết từng fold và độ lệch chuẩn giữa fold
2. **thắng / hoà / thua** — so **từng** buổi ghi với LSTM, trên đủ 1289 buổi của tám người dev

Bảng 2 cần thiết vì hai cấu hình chênh nhau 0.005 điểm trung bình có thể là tốt hơn đều khắp, hoặc thắng đậm vài buổi mà thua nhẹ phần lớn. Trung bình không phân biệt được. Và nó **không tốn thêm giờ GPU**.

Nếu `cv_score` giữa hai kiến trúc chênh ít hơn `cv_std` thì chưa kết luận được — lúc đó mới cần chạy thêm seed cho hai cấu hình sát nhau.


In [ ]:
!python scripts/compare_cv.py --experiment tn1 --so-doi lstm


## 4. Mốc kiểm chứng trên G H I J

`TN1_LSTM.ipynb` đã chạy 3 seed cho LSTM. Giờ chạy 3 seed cho **kiến trúc thắng ở mục 3**.

Train đủ tám người `A B C D E F K L` rồi test 537 buổi ghi của `G H I J` — đúng pipeline bài báo dùng, không phải model fold chỉ train 6 người.

**Đây chưa phải số công bố.** Nếu TN2–TN6 đổi cấu hình thì con số này thành cũ và phải chạy lại ở bước cuối. Nó là mốc kiểm chứng của riêng TN1: cho biết hướng đi có đúng không.

Chạy **một** trong hai ô dưới, khoảng 1 giờ:


In [ ]:
# chạy ô này nếu TCN thắng ở mục 3
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 2


In [ ]:
# chạy ô này nếu DS-TCN thắng ở mục 3
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 2


Gộp ba seed thành `mean ± std`:


In [ ]:
!python scripts/compare_cv.py --experiment tn1_ghij --final


## 5. Cất kết quả

Nén cả `runs/tn1/` và `runs/tn1_ghij/` — bản đã gộp đủ ba cấu hình.


In [ ]:
!python scripts/save_results.py tn1
!python scripts/save_results.py tn1_ghij
